In [ ]:
import re
import itertools
import pandas as pd
import easygui
from rapidfuzz import fuzz

In [ ]:
def text_normalization(output_file_path):
    """Finds close-matching column headings and allows the user to normalize them."""

    if not output_file_path:                                             # (1) Checks whether an Excel file is available.
        print("No Excel file available for text normalization.")
        return None

    # (2) Ensures that the output file has the .xlsx extension.
    output_file_path = str(output_file_path).strip()

    if not output_file_path.lower().endswith(".xlsx"):
        output_file_path += ".xlsx"

    xls = pd.ExcelFile(output_file_path)                                 # (3) Opens the workbook without loading its full data.

    headings = [col for sheet in xls.sheet_names                         # (4) Extracts only the column headings from all worksheets.
                for col in pd.read_excel(xls, sheet_name=sheet, nrows=0).columns]

    prepared = {col: (                                                   # (5) Prepares each heading once for comparison.
        re.sub(r"[\W_]+", "", str(col)).lower(),
        re.findall(r"\d+", str(col))
    ) for col in headings}

    parent = {col: col for col in headings}                              # (6) Initializes grouping of close-matching headings.

    for a, b in itertools.combinations(headings, 2):                     # (7) Compares every unique pair of headings.
        text_a, numbers_a = prepared[a]
        text_b, numbers_b = prepared[b]

        if (a != b
            and numbers_a == numbers_b
            and (text_a in text_b
                 or text_b in text_a
                 or fuzz.ratio(text_a, text_b) >= 85)):

            root_a = a
            while parent[root_a] != root_a:
                root_a = parent[root_a]

            root_b = b
            while parent[root_b] != root_b:
                root_b = parent[root_b]

            parent[root_b] = root_a                                     # (8) Joins close and transitive matches.

    groups = {}                                                          # (9) Collects connected headings into match groups.

    for col in headings:
        root = col
        while parent[root] != root:
            root = parent[root]

        groups.setdefault(root, set()).add(col)

    groups = [sorted(group) for group in groups.values() if len(group) > 1]  # (10) Keeps groups containing different headings.

    rename = {}                                                          # (11) Stores user-approved normalized headings.

    for group in groups:

        suggested = re.sub(                                              # (12) Creates a readable preset from the longest variant.
            r"(?<=[A-Za-z])(?=\d)|(?<=\d)(?=[A-Za-z])", " ",
            max(group, key=len).replace("_", " ")
        ).title()

        normalized = easygui.enterbox(                                   # (13) Displays each group with an editable suggested name.
            msg="Close matching parameters found:\n\n"
                + "\n".join(f"{i}. {col}" for i, col in enumerate(group, 1))
                + "\n\nEnter the normalized parameter name:",
            title="TEXT NORMALIZATION",
            default=suggested
        )

        # Cancel or closing the dialog returns None.
        # In that case, this group is deliberately left unchanged.
        if normalized is None:
            continue

        normalized = normalized.strip()

        # A blank entry also means: retain the original headings.
        if not normalized:
            continue

        rename.update(dict.fromkeys(group, normalized))                   # (14) Stores only explicitly approved normalization.

    xls.close()                                                           # (15) Releases the workbook before rewriting it.

    if rename:                                                            # (16) Loads the full workbook only when changes are required.
        all_sheets = pd.read_excel(output_file_path, sheet_name=None)

        with pd.ExcelWriter(
            output_file_path,
            engine="openpyxl",
            mode="w"
        ) as writer:

            for sheet, df in all_sheets.items():
                df.rename(columns=rename).to_excel(
                    writer,
                    sheet_name=sheet,
                    index=False
                )

    print("=" * 90)                                                       # (17) Displays the completion message.
    print(f"✓ Text Normalization Complete ({len(rename)} headings renamed)")
    print("=" * 90)

    return output_file_path                                               # (18) Returns the path including the .xlsx extension.